## 数据集简要说明

ETT = Electricity Transformer Temperature，电力变压器温度数据集，时间范围：2016-07 ~ 2018-07，一共 7 个变量：
`date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT`

- OT：油温（预测目标）
- HUFL/HULL/MUFL/MULL/LUFL/LULL：高低压侧负载特征

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [3]:
# ---------------------- 数据集类：滑动窗口构造时序样本 ----------------------
class ETTDataset(Dataset):
    def __init__(self, data, lookback=96, pred_len=24):
        """
        data: 归一化后的数组 [T, 7]
        lookback: 用前面多少步历史
        pred_len: 预测未来多少步 OT
        """
        self.lookback = lookback
        self.pred_len = pred_len
        self.data = data
        self.samples = []
        self.labels = []

        T = len(data)
        for i in range(T - lookback - pred_len + 1):
            x_seq = data[i:i+lookback, :]       # [lookback, 7]
            y_seq = data[i+lookback:i+lookback+pred_len, -1:]  # 只预测最后一列 OT
            self.samples.append(x_seq)
            self.labels.append(y_seq.squeeze(-1))

        self.samples = np.array(self.samples, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.samples[idx])
        y = torch.from_numpy(self.labels[idx])
        return x, y

In [4]:
# ---------------------- 数据加载、归一化、划分 ----------------------
def load_ett(csv_path="ETTh1.csv", lookback=96, pred_len=24):
    df = pd.read_csv(csv_path)
    # 去掉date，只保留7个数值特征
    raw_data = df[["HUFL","HULL","MUFL","MULL","LUFL","LULL","OT"]].values

    # 时序划分：训练/验证/测试（标准划分）
    train_size = int(0.7 * len(raw_data))
    val_size = int(0.1 * len(raw_data))
    test_size = len(raw_data) - train_size - val_size

    train_data = raw_data[:train_size]
    val_data   = raw_data[train_size:train_size+val_size]
    test_data  = raw_data[train_size+val_size:]

    # 用训练集统计量做归一化（重要！不能泄露测试集信息）
    mean = train_data.mean(axis=0)
    std  = train_data.std(axis=0)
    train_data = (train_data - mean) / std
    val_data   = (val_data - mean) / std
    test_data  = (test_data - mean) / std

    train_ds = ETTDataset(train_data, lookback, pred_len)
    val_ds   = ETTDataset(val_data, lookback, pred_len)
    test_ds  = ETTDataset(test_data, lookback, pred_len)

    return train_ds, val_ds, test_ds, mean, std

In [6]:
lookback = 96
pred_len = 24
batch_size = 32
lr = 1e-3
epochs = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds, val_ds, test_ds, mean, std = load_ett("./Datasets/ETTh1.csv", lookback, pred_len)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [7]:
len(train_ds)

12075

In [ ]:
train_ds[0][0].shape, train_ds[0][1].shape

(torch.Size([96, 7]), torch.Size([24]))

In [9]:
train_ds[0][1]

tensor([1.1575, 1.1575, 1.1659, 1.5114, 1.6968, 1.8064, 1.9749, 1.8484, 1.6800,
        1.6968, 1.9917, 1.7726, 1.1238, 1.3008, 1.4946, 1.7136, 1.8822, 2.0169,
        1.8231, 1.7642, 1.8147, 1.8484, 1.7558, 1.6968])

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=64, num_layers=2, pred_len=24):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.linear = nn.Linear(hidden_dim, pred_len)

    def forward(self, x):
        # x: [B, lookback, 7]
        out, _ = self.lstm(x)
        last_h = out[:, -1, :]  # 取最后一个时间步的hidden state
        pred = self.linear(last_h)
        return pred


